# LangSAT Kaggle Reproduce

In [ ]:
from kaggle_secrets import UserSecretsClient
import os, sys, subprocess, glob, json, pathlib, shutil

REPO_URL = "https://github.com/reikfowo17/LangSAT.git"
REPO_DIR = "/kaggle/working/repo"
REPO_BRANCH = "main"
DATA_DIR = "/kaggle/input/datasets/heon29/uf20-91"
OUTPUT_DIR = "/kaggle/working/results"
SATFEATPY_DIR = "/kaggle/working/SATfeatPy"

github_token = None
try:
    github_token = UserSecretsClient().get_secret("GITHUB_TOKEN")
except Exception:
    github_token = None

subprocess.run(["rm", "-rf", REPO_DIR], check=False)
clone_url = REPO_URL
if github_token:
    clone_url = REPO_URL.replace("https://", f"https://x-access-token:{github_token}@")
subprocess.run(["git", "clone", "--branch", REPO_BRANCH, "--single-branch", clone_url, REPO_DIR], check=True)
subprocess.run(["git", "-C", REPO_DIR, "remote", "set-url", "origin", REPO_URL], check=False)
subprocess.run(["git", "-C", REPO_DIR, "status", "--short", "--branch"], check=False)
subprocess.run(["git", "-C", REPO_DIR, "log", "--oneline", "-1"], check=False)

# SATfeatPy provides the SATzilla-style 48 global features described in the paper.
# Paper-like runs require SATfeatPy; feature extraction should fail loudly if it is unavailable.
if not os.path.exists(SATFEATPY_DIR):
    subprocess.run([
        "git", "clone", "--depth", "1",
        "https://github.com/bprovanbessell/SATfeatPy.git",
        SATFEATPY_DIR,
    ], check=False)

sys.path.insert(0, f"{REPO_DIR}/src")
os.makedirs(OUTPUT_DIR, exist_ok=True)
os.environ["LANGSAT_DATA_DIR"] = DATA_DIR
os.environ["LANGSAT_OUTPUT_DIR"] = OUTPUT_DIR
os.environ["LANGSAT_MODEL_PATH"] = f"{OUTPUT_DIR}/smartsat_model"
os.environ["LANGSAT_FEATURE_BACKEND"] = "satfeatpy"
os.environ["LANGSAT_SATFEATPY_DIR"] = SATFEATPY_DIR
os.environ["LANGSAT_FEATURE_CACHE_DIR"] = f"{OUTPUT_DIR}/feature_cache"
# Full SATzilla features 41-48 require ubcsat local-search probing and can be slow.
# Keep this off for the main Kaggle run; set to 1 for a smaller ablation if needed.
os.environ["LANGSAT_SATFEATPY_FULL_LOCAL_SEARCH"] = "0"
# Keep reported metrics raw by default. Set LANGSAT_TIME_SCALE manually only
# if you need a separate hardware-normalized comparison table.
os.environ["LANGSAT_REPORT_SCALE_TO_PAPER"] = "0"
os.environ["LANGSAT_POLICY_MODE"] = "rl"
# Paper-like reward: +1 satisfied clause, -1 unsatisfied clause.
os.environ["LANGSAT_REWARD_MODE"] = "paper"
os.environ["LANGSAT_STEP_PENALTY"] = "0"
os.environ["LANGSAT_CONFLICT_PENALTY"] = "0"
os.environ["LANGSAT_REQUIRE_SATFEATPY"] = "1"
# Use PySAT only after the Python CDCL budget is hit, and report the Python budget/search time
# instead of the fast Minisat22 fallback time in paper-like metrics.
os.environ["LANGSAT_USE_PYSAT"] = "1"
os.environ["LANGSAT_BASELINE_USE_PYSAT"] = "1"
os.environ["LANGSAT_SMARTSAT_USE_PYSAT"] = "1"
os.environ["LANGSAT_PYSAT_TIME_MODE"] = "budget"
# Paper-like metric: compare total solving time, including PPO inference overhead.
os.environ["LANGSAT_USE_SEARCH_TIME"] = "0"

print("Repo:", REPO_DIR)
print("Branch:", REPO_BRANCH)
print("Data:", DATA_DIR)
print("Output:", OUTPUT_DIR)
print("SATfeatPy:", SATFEATPY_DIR, "exists=", os.path.exists(SATFEATPY_DIR))
print("CNF files:", len(glob.glob(f"{DATA_DIR}/**/*.cnf", recursive=True)))

In [ ]:
!pip install -q -r /kaggle/working/repo/requirements.txt
!pip install -q "shimmy>=2.0"
!pip install -q "python-louvain>=0.16" "powerlaw>=1.5"

## Feature backend smoke test

In [ ]:
import sys, glob, os
sys.path.insert(0, f"{REPO_DIR}/src")
from satfeat_adapter import BACKEND_USAGE, extract_sat_features

sample_cnf = sorted(glob.glob(f"{DATA_DIR}/**/*.cnf", recursive=True))[0]
features = extract_sat_features(sample_cnf)
print("Sample CNF:", os.path.basename(sample_cnf))
print("Feature shape:", features.shape)
print("Feature min/max:", float(features.min()), float(features.max()))
print("Feature first 8:", features[:8])
print("Feature backend usage:", BACKEND_USAGE)
assert BACKEND_USAGE["fallback"] == 0, "Paper-like run requires SATfeatPy; fallback features were used."


## Quick smoke test

In [ ]:
import glob, os, sys
sys.path.insert(0, f"{REPO_DIR}/src")
from cdcl_baseline import solve_file
from training_pipeline import load_and_split_dataset

train_files, test_files = load_and_split_dataset(DATA_DIR, train_ratio=0.8)
for f in test_files[:3]:
    sat, t = solve_file(f)
    print(os.path.basename(f), "SAT" if sat else "UNSAT", f"{t:.6f}s")

## Train

In [ ]:
os.environ["LANGSAT_TOTAL_STEPS"] = "100000"

from training_pipeline import train_smartsat, plot_reward_curve
model, callback = train_smartsat(train_files)
plot_reward_curve(callback)

## Evaluate

In [ ]:
from evaluate import evaluate, compute_metrics, plot_solving_times, plot_time_distribution

df = evaluate(test_files, model)
metrics = compute_metrics(df)
plot_solving_times(df, metrics)
plot_time_distribution(df)
metrics

## Summary table

In [ ]:
with open(f"{OUTPUT_DIR}/metrics.json") as f:
    m = json.load(f)

print("KET QUA PAPER-LIKE REPRODUCE")
print(f"{'Metric':<25} {'Reproduce':>12} {'Paper':>12}")
print(f"{'-'*51}")
print(f"{'Win Rate (%)':<25} {m['win_rate_pct']:>12.2f} {'~53.00':>12}")
print(f"{'Median SmartSAT (s)':<25} {m['median_smartsat']:>12.4f} {'~1.02':>12}")
print(f"{'Median Baseline (s)':<25} {m['median_baseline']:>12.4f} {'~1.02':>12}")
print(f"{'SAT Rate SmartSAT (%)':<25} {m['sat_rate_smartsat']:>12.2f} {'100.00':>12}")
print(f"{'SAT Rate Baseline (%)':<25} {m['sat_rate_baseline']:>12.2f} {'100.00':>12}")
print(f"{'Raw Median SmartSAT':<25} {m['median_smartsat_raw']:>12.6f} {'':>12}")
print(f"{'Raw Median Baseline':<25} {m['median_baseline_raw']:>12.6f} {'':>12}")
print(f"{'Time Scale':<25} {m['time_scale']:>12.6f} {'1.0 raw':>12}")
print(f"{'Raw Search SmartSAT':<25} {m['median_smartsat_search_raw']:>12.6f} {'':>12}")
print(f"{'Raw Policy Overhead':<25} {m['median_policy_time_raw']:>12.6f} {'':>12}")
print(f"{'Policy Time / Call':<25} {m.get('median_policy_time_per_call_raw', 0):>12.8f} {'':>12}")
print(f"{'Policy Mode':<25} {m['policy_mode']:>12} {'':>12}")
print(f"{'Invalid Action (%)':<25} {m.get('median_invalid_action_rate_pct', 0):>12.2f} {'0 target':>12}")
print(f"{'Decision Ratio ST/BSL':<25} {m.get('median_decision_ratio_st_over_bsl', 0):>12.4f} {'<1 better':>12}")
print(f"{'Conflict Ratio ST/BSL':<25} {m.get('median_conflict_ratio_st_over_bsl', 0):>12.4f} {'<1 better':>12}")
print(f"{'PySAT Time Mode':<25} {m.get('pysat_time_mode', ''):>12} {'budget':>12}")
print(f"{'Feature Backend':<25} {m.get('feature_backend', ''):>12} {'':>12}")
print(f"{'Feature Usage':<25} {str(m.get('feature_backend_usage', {})):>12} {'':>12}")
print(f"{'SATfeatPy Local Search':<25} {str(m.get('satfeatpy_full_local_search', False)):>12} {'':>12}")

print("\nSaved files:")
for path in sorted(glob.glob(f"{OUTPUT_DIR}/*")):
    print(path)